# ISO 52016 Calibration against EnergyPlus 24.1.0
## Apt 305, 50 Barry St, Carlton, Melbourne

Finds the ISO 52016 configuration (ventilation rate / H_ve coefficient) that
best matches a full EnergyPlus heat-balance simulation.

**Workflow:**
1. Run EnergyPlus with hourly zone-temperature + energy outputs → ground-truth CSV
2. Define candidate ISO configurations (valid ventilation types only)
3. Run each candidate through ISO 52016, score vs EnergyPlus
4. Report ranked results and best calibrated configuration

In [ ]:
!pip install -q pybuildingenergy requests pandas matplotlib numpy

In [ ]:
import os, sys, copy, re, glob, shutil, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WEATHER_SOURCE = "epw"
FLOOR_AREA     = 5.0 * 4.0   # m²


In [ ]:
def build_bui():
    """Returns a fresh copy of the Apt 305 building dictionary (no shared state)."""

    # --- Construction U-values & thermal capacity — Australian BCA 2006 minimum-spec ---
    U_EXT_WALL  = 1.00   # brick veneer / precast w/ R1.0 insulation
    U_INT_WALL  = 2.50   # concrete block + plasterboard, no insulation
    U_INT_SLAB  = 1.80   # 200 mm concrete intermediate floor
    U_WINDOW    = 5.40   # aluminium-frame single glazing
    G_WINDOW    = 0.65   # SHGC of clear single glazing

    ABS_EXT_WALL = 0.75  # dark red brick
    ABS_INT      = 0.0

    C_EXT_WALL = 450_000   # heavy concrete external wall, J/m2K
    C_INT_WALL = 330_000   # concrete-block partition
    C_INT_SLAB = 480_000   # 200 mm concrete slab
    C_WINDOW   = 0

    # --- Geometry ---
    LEN_NS  = 5.0   # N-S length, m (west facade width, along Barry St)
    LEN_EW  = 4.0   # E-W depth, m
    HEIGHT  = 2.7   # ceiling height, m

    FLOOR_AREA = LEN_NS * LEN_EW
    VOLUME     = FLOOR_AREA * HEIGHT

    A_WEST_GROSS  = LEN_NS * HEIGHT
    A_EAST_GROSS  = LEN_NS * HEIGHT
    A_NORTH_GROSS = LEN_EW * HEIGHT
    A_SOUTH_GROSS = LEN_EW * HEIGHT

    WIN_WIDTH_FIXED,    WIN_HEIGHT_FIXED    = 0.9, 0.9
    WIN_WIDTH_OPERABLE, WIN_HEIGHT_OPERABLE = 0.9, 0.9
    A_WINDOW_FIXED    = WIN_WIDTH_FIXED * WIN_HEIGHT_FIXED
    A_WINDOW_OPERABLE = WIN_WIDTH_OPERABLE * WIN_HEIGHT_OPERABLE
    A_WINDOW_TOTAL    = A_WINDOW_FIXED + A_WINDOW_OPERABLE
    A_WEST_OPAQUE     = A_WEST_GROSS - A_WINDOW_TOTAL

    bui = {
        "building": {
            "name": "Apt_305_50_Barry_St_Carlton",
            "azimuth_relative_to_true_north": 0,
            "latitude":  -37.800,
            "longitude": 144.968,
            "exposed_perimeter": 0,
            "height": HEIGHT,
            "wall_thickness": 0.20,
            "n_floors": 1,
            "building_type_class": "Residential_apartment",
            "adj_zones_present": True,
            "number_adj_zone": 5,
            "net_floor_area": FLOOR_AREA,
            "construction_class": "class_iii",
            "construction_year": "2006-today",
            "country": "Australia",
        },
        "adjacent_zones": [
            {
                "name": "apt_above",
                "orientation_zone": {"azimuth": 270.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_below",
                "orientation_zone": {"azimuth": 270.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_EXT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_north",
                "orientation_zone": {"azimuth": 0.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "apt_south",
                "orientation_zone": {"azimuth": 180.0},
                "area_facade_elements": np.array([A_WEST_GROSS, A_NORTH_GROSS, A_EAST_GROSS, A_SOUTH_GROSS, FLOOR_AREA, FLOOR_AREA]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_WALL, U_INT_SLAB, U_INT_SLAB]),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": VOLUME,
                "building_type_class": "Residential_apartment",
                "a_use": FLOOR_AREA,
            },
            {
                "name": "corridor",
                "orientation_zone": {"azimuth": 90.0},
                "area_facade_elements": np.array([81.0, 5.4, 81.0, 5.4, 60.0, 60.0]),
                "typology_elements": ["OP", "OP", "OP", "OP", "OP", "OP"],
                "transmittance_U_elements": np.array([U_INT_WALL] * 6),
                "orientation_elements": np.array(["WV", "NV", "EV", "SV", "HOR", "HOR"]),
                "volume": 162.0,
                "building_type_class": "Residential_apartment",
                "a_use": 60.0,
            },
        ],
        "building_surface": [
            {
                "name": "West exterior wall (opaque)", "type": "opaque", "area": A_WEST_OPAQUE,
                "sky_view_factor": 0.5, "u_value": U_EXT_WALL, "solar_absorptance": ABS_EXT_WALL,
                "thermal_capacity": C_EXT_WALL, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": HEIGHT, "length": LEN_NS,
            },
            {
                "name": "North wall to Apt 306", "type": "opaque", "area": A_NORTH_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 0.0, "tilt": 90.0},
                "name_adj_zone": "apt_north", "height": HEIGHT, "length": LEN_EW,
            },
            {
                "name": "South wall to Apt 304", "type": "opaque", "area": A_SOUTH_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 180.0, "tilt": 90.0},
                "name_adj_zone": "apt_south", "height": HEIGHT, "length": LEN_EW,
            },
            {
                "name": "East wall to corridor", "type": "opaque", "area": A_EAST_GROSS,
                "sky_view_factor": 0.0, "u_value": U_INT_WALL, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_WALL, "orientation": {"azimuth": 90.0, "tilt": 90.0},
                "name_adj_zone": "corridor", "height": HEIGHT, "length": LEN_NS,
            },
            {
                "name": "Floor to Apt 205", "type": "opaque", "area": FLOOR_AREA,
                "sky_view_factor": 0.0, "u_value": U_INT_SLAB, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_SLAB, "orientation": {"azimuth": 0.0, "tilt": 0.0},
                "name_adj_zone": "apt_below", "height": LEN_NS, "length": LEN_EW,
            },
            {
                "name": "Ceiling to Apt 405", "type": "opaque", "area": FLOOR_AREA,
                "sky_view_factor": 0.0, "u_value": U_INT_SLAB, "solar_absorptance": ABS_INT,
                "thermal_capacity": C_INT_SLAB, "orientation": {"azimuth": 0.0, "tilt": 0.0},
                "name_adj_zone": "apt_above", "height": LEN_NS, "length": LEN_EW,
            },
            {
                "name": "West window — fixed", "type": "transparent", "area": A_WINDOW_FIXED,
                "sky_view_factor": 0.5, "u_value": U_WINDOW, "solar_absorptance": 0.5,
                "thermal_capacity": C_WINDOW, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": WIN_HEIGHT_FIXED, "g_value": G_WINDOW,
                "width": WIN_WIDTH_FIXED, "parapet": 1.0, "shading": True,
                "shading_type": "horizontal_overhang", "width_or_distance_of_shading_elements": 0.05,
                "overhang_proprieties": {"width_of_horizontal_overhangs": 0.25},
            },
            {
                "name": "West window — operable", "type": "transparent", "area": A_WINDOW_OPERABLE,
                "sky_view_factor": 0.5, "u_value": U_WINDOW, "solar_absorptance": 0.5,
                "thermal_capacity": C_WINDOW, "orientation": {"azimuth": 270.0, "tilt": 90.0},
                "name_adj_zone": None, "height": WIN_HEIGHT_OPERABLE, "g_value": G_WINDOW,
                "width": WIN_WIDTH_OPERABLE, "parapet": 1.0, "shading": True,
                "shading_type": "horizontal_overhang", "width_or_distance_of_shading_elements": 0.05,
                "overhang_proprieties": {"width_of_horizontal_overhangs": 0.25},
            },
        ],
        "units": {
            "area": "m\u00b2", "u_value": "W/m\u00b2K", "thermal_capacity": "J/m\u00b2K",
            "azimuth": "degrees (0=N, 90=E, 180=S, 270=W)", "tilt": "degrees (0=horizontal, 90=vertical)",
            "internal_gain": "W/m\u00b2", "HVAC_profile": "0: off, 1: on",
        },
        "building_parameters": {
            "temperature_setpoints": {
                # Fixed setpoints, as used in copy_of_pybuildingenergy_my_house.ipynb.
                # The modified-engine cell overrides these with NCC zone-derived setpoints.
                "heating_setpoint": 18.0,
                "heating_setback":  15.0,
                "cooling_setpoint": 26.0,
                "cooling_setback":  28.0,
                "units": "\u00b0C",
            },
            "system_capacities": {
                "heating_capacity": 10_000_000.0,
                "cooling_capacity": 10_000_000.0,
                "units": "W",
            },
            "ventilation": {
                "ventilation_type": "occupancy",
                "flow_rate_per_person": 2.0,
                "units": "l/(s m\u00b2)",
                "custom_heat_transfer_coefficient_ventilation": None,
            },
            "internal_gains": [
                {
                    "name": "occupants", "full_load": 8.0,
                    "weekday": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.5,0.4,0.5,0.5,0.5,0.4,0.5,0.5,0.5,0.5,0.5,1.0,1.0,1.0,1.0],
                    "weekend": [1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.8,0.7,0.7,0.7,0.7,0.5,0.5,0.7,0.8,1.0,1.0,1.0,1.0,1.0,1.0],
                },
                {
                    "name": "appliances", "full_load": 25.0,
                    "weekday": [0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.2,0.2,0.2,0.2,0.3,0.3,0.4,1.0,0.6,0.4,0.2],
                    "weekend": [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.2,0.3,0.4,0.3,0.3,0.4,0.3,0.3,0.3,0.3,0.4,0.4,0.5,1.0,0.6,0.4,0.2],
                },
                {
                    "name": "lighting", "full_load": 3.0,
                    "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.3,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                    "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.1,0.3,0.3,0.2,0.2,0.2,0.2,0.2,0.2,0.3,0.5,0.8,0.8,0.8,0.7,0.4,0.1],
                },
            ],
            "construction": {
                "wall_thickness": 0.20,
                "thermal_bridges": 1.5,
                "units": "m (thickness), W/mK (thermal bridges)",
            },
            "climate_parameters": {"coldest_month": 7, "units": "1-12 (January-December)"},
            "heating_profile": {
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0],
            },
            "cooling_profile": {
                "weekday": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
                "weekend": [0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0],
            },
            "ventilation_profile": {
                "weekday": [1.0] * 24,
                "weekend": [1.0] * 24,
            },
        },
    }
    return bui

FLOOR_AREA = build_bui()["building"]["net_floor_area"]
print(f"Floor area: {FLOOR_AREA} m2")


In [ ]:
from pybuildingenergy.source.check_input import sanitize_and_validate_BUI as orig_sanitize
from pybuildingenergy.source.utils import ISO52016 as OrigISO52016
print("ISO 52016 engine loaded:", OrigISO52016.__module__)


In [ ]:
# ── EnergyPlus 24.1.0 — install via tar.gz (no interactive prompt) ───────────
import os, subprocess, sys, requests

EP_VERSION = "24.1.0"
EP_SHA     = "9d7789a3ac"
EP_DIR     = "/opt/energyplus"

if not os.path.exists(EP_DIR):
    tar_name = f"EnergyPlus-{EP_VERSION}-{EP_SHA}-Linux-Ubuntu22.04-x86_64.tar.gz"
    tar_url  = (
        f"https://github.com/NREL/EnergyPlus/releases/download/"
        f"v{EP_VERSION}/{tar_name}"
    )
    tar_path = f"/tmp/{tar_name}"

    print(f"Downloading EnergyPlus {EP_VERSION} (tar.gz) …")
    subprocess.run(["wget", "-q", "--show-progress", "-O", tar_path, tar_url], check=True)

    print("Extracting …")
    os.makedirs(EP_DIR, exist_ok=True)
    # --strip-components=1 removes the top-level folder inside the archive
    subprocess.run(
        ["tar", "-xzf", tar_path, "--strip-components=1", "-C", EP_DIR],
        check=True,
    )
    os.remove(tar_path)
    print(f"Installed → {EP_DIR}")
else:
    print(f"EnergyPlus already at {EP_DIR}")

# Confirm the binary exists before continuing
ep_bin = os.path.join(EP_DIR, "energyplus")
assert os.path.exists(ep_bin), f"energyplus binary not found in {EP_DIR}"
print(f"Binary OK: {ep_bin}")

if EP_DIR not in sys.path:
    sys.path.insert(0, EP_DIR)

# ── PVGIS TMY → EPW  (same weather source as the ISO 52016 simulations) ──────
EPW_PATH = "/content/Melbourne_PVGIS_TMY.epw"
if not os.path.exists(EPW_PATH):
    print("Downloading PVGIS TMY as EPW …")
    pvgis_url = (
        "https://re.jrc.ec.europa.eu/api/v5_2/tmy"
        "?lat=-37.800&lon=144.968&outputformat=epw"
    )
    r = requests.get(pvgis_url, timeout=120)
    r.raise_for_status()
    with open(EPW_PATH, "wb") as f:
        f.write(r.content)
    print(f"Saved EPW → {EPW_PATH}  ({len(r.content)//1024} kB)")
else:
    print(f"EPW already at {EPW_PATH}")

with open(EPW_PATH) as f:
    first = f.readline().strip()
print(f"EPW header: {first[:80]}")

In [ ]:
def _sched_frac(name, wd, we):
    rows = ["Schedule:Compact,\n  {},\n  Fraction,\n  Through: 12/31,".format(name),
            "  For: Weekdays,"]
    for h, v in enumerate(wd, 1):
        rows.append("  Until: {:02d}:00,{},".format(h, v))
    rows.append("  For: AllOtherDays,")   # covers weekends + all design-day types
    for h, v in enumerate(we, 1):
        rows.append("  Until: {:02d}:00,{}{}".format(h, v, "," if h < 24 else ";"))
    return "\n".join(rows)

def _sched_temp(name, wd, we):
    rows = ["Schedule:Compact,\n  {},\n  Temperature,\n  Through: 12/31,".format(name),
            "  For: Weekdays,"]
    for h, v in enumerate(wd, 1):
        rows.append("  Until: {:02d}:00,{},".format(h, v))
    rows.append("  For: AllOtherDays,")
    for h, v in enumerate(we, 1):
        rows.append("  Until: {:02d}:00,{}{}".format(h, v, "," if h < 24 else ";"))
    return "\n".join(rows)

def build_ep_idf(heating_sp=18.0, cooling_sp=26.0, heating_sb=15.0, cooling_sb=28.0):
    """
    EnergyPlus IDF for Apt 305 mirroring build_bui() inputs.
    Material:NoMass R-values back-calculated to match assembly U-values.
    5 neighbour surfaces fixed at 20 C via OtherSideCoefficients.
    ZoneHVAC:IdealLoadsAirSystem (unlimited capacity).
    Ventilation: 0.002 m3/s/m2.
    """
    bui  = build_bui()
    bp   = bui["building_parameters"]
    ig   = {g["name"]: g for g in bp["internal_gains"]}

    occ_wd = ig["occupants"]["weekday"];  occ_we = ig["occupants"]["weekend"]
    app_wd = ig["appliances"]["weekday"]; app_we = ig["appliances"]["weekend"]
    lit_wd = ig["lighting"]["weekday"];   lit_we = ig["lighting"]["weekend"]
    h_wd = bp["heating_profile"]["weekday"]; h_we = bp["heating_profile"]["weekend"]
    c_wd = bp["cooling_profile"]["weekday"]; c_we = bp["cooling_profile"]["weekend"]

    hsp_wd = [heating_sp if v > 0 else heating_sb for v in h_wd]
    hsp_we = [heating_sp if v > 0 else heating_sb for v in h_we]
    csp_wd = [cooling_sp if v > 0 else cooling_sb for v in c_wd]
    csp_we = [cooling_sp if v > 0 else cooling_sb for v in c_we]

    R_ext = round(max(0.001, 1/1.00 - 0.13 - 0.04), 4)
    R_int = round(max(0.001, 1/2.50 - 0.13 - 0.13), 4)
    R_slb = round(max(0.001, 1/1.80 - 0.13 - 0.13), 4)
    LEN_NS = 5.0; LEN_EW = 4.0; H = 2.7

    schedules = "\n\n".join([
        _sched_frac("Occ_Sched",  occ_wd, occ_we),
        _sched_frac("App_Sched",  app_wd, app_we),
        _sched_frac("Lit_Sched",  lit_wd, lit_we),
        _sched_temp("Heat_SP",    hsp_wd, hsp_we),
        _sched_temp("Cool_SP",    csp_wd, csp_we),
        "Schedule:Compact,\n  Always1,\n  Fraction,\n  Through: 12/31,"
        "\n  For: AllDays,\n  Until: 24:00,1.0;",
        "Schedule:Compact,\n  Activity_W,\n  Any Number,\n  Through: 12/31,"
        "\n  For: AllDays,\n  Until: 24:00,160.0;",
    ])

    parts = []

    parts.append(
        "  Version, 24.1;\n\n"
        "  SimulationControl, Yes, No, No, No, Yes;\n\n"
        "  Building,\n"
        "    Apt_305_50_Barry_St_Carlton,\n"
        "    0.0, Suburbs, 0.04, 0.004,\n"
        "    FullInteriorAndExterior, 25, 6;\n\n"
        "  Timestep, 6;\n\n"
        "  GlobalGeometryRules,\n"
        "    UpperLeftCorner, CounterClockWise, World;\n\n"
        "  Site:Location,\n"
        "    Melbourne_VIC_AUS,\n"
        "    -37.800, 144.968, 10.0, 31.0;\n\n"
        "  RunPeriod,\n"
        "    FullYear,\n"
        "    1, 1, , 12, 31, ,\n"
        "    Sunday, Yes, Yes, No, Yes, Yes;\n\n"
        "  ScheduleTypeLimits, Fraction, 0.0, 1.0, Continuous, Dimensionless;\n"
        "  ScheduleTypeLimits, Temperature, -100, 200, Continuous, Temperature;\n"
        "  ScheduleTypeLimits, Any Number, -1e10, 1e10, Continuous;"
    )

    parts.append(schedules)

    parts.append(
        "  Zone, Apt305, 0.0, 0.0, 0.0, 0.0, 1, 1, {}, {}, {};".format(
            H, LEN_EW*LEN_NS*H, LEN_EW*LEN_NS)
    )

    parts.append(
        "  Material:NoMass, Mat_ExtWall, MediumRough,  {}, 0.9, 0.75, 0.75;\n"
        "  Material:NoMass, Mat_IntWall, MediumSmooth, {}, 0.9, 0.0,  0.0;\n"
        "  Material:NoMass, Mat_IntSlab, MediumSmooth, {}, 0.9, 0.0,  0.0;\n"
        "  WindowMaterial:SimpleGlazingSystem, Mat_Window, 5.40, 0.65;\n\n"
        "  Construction, Con_ExtWall, Mat_ExtWall;\n"
        "  Construction, Con_IntWall, Mat_IntWall;\n"
        "  Construction, Con_IntSlab, Mat_IntSlab;\n"
        "  Construction, Con_Window,  Mat_Window;".format(R_ext, R_int, R_slb)
    )

    parts.append(
        "  SurfaceProperty:OtherSideCoefficients,\n"
        "    OSC_Adj20C,\n"
        "    8.0,\n"
        "    20.0,\n"
        "    1.0, 0.0, 0.0, 0.0, 0.0;"
    )

    parts.append(
        "  BuildingSurface:Detailed,\n"
        "    WestWall, Wall, Con_ExtWall, Apt305, ,\n"
        "    Outdoors, , SunExposed, WindExposed, , 4,\n"
        "    0.0, {LN}, {H},\n"
        "    0.0, {LN}, 0.0,\n"
        "    0.0, 0.0, 0.0,\n"
        "    0.0, 0.0, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    NorthWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj20C, NoSun, NoWind, , 4,\n"
        "    {LE}, {LN}, {H},\n"
        "    {LE}, {LN}, 0.0,\n"
        "    0.0, {LN}, 0.0,\n"
        "    0.0, {LN}, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    EastWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj20C, NoSun, NoWind, , 4,\n"
        "    {LE}, 0.0, {H},\n"
        "    {LE}, 0.0, 0.0,\n"
        "    {LE}, {LN}, 0.0,\n"
        "    {LE}, {LN}, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    SouthWall, Wall, Con_IntWall, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj20C, NoSun, NoWind, , 4,\n"
        "    0.0, 0.0, {H},\n"
        "    0.0, 0.0, 0.0,\n"
        "    {LE}, 0.0, 0.0,\n"
        "    {LE}, 0.0, {H};\n\n"
        "  BuildingSurface:Detailed,\n"
        "    FloorSlab, Floor, Con_IntSlab, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj20C, NoSun, NoWind, , 4,\n"
        "    0.0, 0.0, 0.0,\n"
        "    0.0, {LN}, 0.0,\n"
        "    {LE}, {LN}, 0.0,\n"
        "    {LE}, 0.0, 0.0;\n\n"
        "  BuildingSurface:Detailed,\n"
        "    CeilingSlab, Ceiling, Con_IntSlab, Apt305, ,\n"
        "    OtherSideCoefficients, OSC_Adj20C, NoSun, NoWind, , 4,\n"
        "    0.0, 0.0, {H},\n"
        "    {LE}, 0.0, {H},\n"
        "    {LE}, {LN}, {H},\n"
        "    0.0, {LN}, {H};\n\n"
        "  FenestrationSurface:Detailed,\n"
        "    WestWin_Fixed, Window, Con_Window, WestWall, , , , 1.0, 4,\n"
        "    0.0, {y1h}, 1.9,  0.0, {y1h}, 1.0,\n"
        "    0.0, {y1l}, 1.0,  0.0, {y1l}, 1.9;\n\n"
        "  FenestrationSurface:Detailed,\n"
        "    WestWin_Operable, Window, Con_Window, WestWall, , , , 1.0, 4,\n"
        "    0.0, {y2h}, 1.9,  0.0, {y2h}, 1.0,\n"
        "    0.0, {y2l}, 1.0,  0.0, {y2l}, 1.9;".format(
            H=H, LN=LEN_NS, LE=LEN_EW,
            y1h=LEN_NS-1.0, y1l=LEN_NS-1.9,
            y2h=LEN_NS-2.5, y2l=LEN_NS-3.4,
        )
    )

    parts.append(
        # People: Method, LightingLevel(blank), PeoplePerArea, ZoneAreaPerPerson(blank),
        #         FracRadiant, SensibleHeatFrac(blank), ActivitySchedule
        "  People,\n"
        "    Apt305_People, Apt305, Occ_Sched,\n"
        "    People/Area, , 0.05, , 0.3, , Activity_W;\n\n"
        # ElectricEquipment: Method, DesignLevel(blank), W/Area, W/Person(blank),
        #   FracLatent, FracRadiant, FracLost
        "  ElectricEquipment,\n"
        "    Apt305_Appliances, Apt305, App_Sched,\n"
        "    Watts/Area, , 25.0, , 0.0, 0.5, 0.0;\n\n"
        # Lights: Method, LightingLevel(blank), W/Area, W/Person(blank),
        #   ReturnAirFrac, FracRadiant, FracVisible
        "  Lights,\n"
        "    Apt305_Lights, Apt305, Lit_Sched,\n"
        "    Watts/Area, , 3.0, , 0.0, 0.72, 0.18;"
    )

    parts.append(
        "  DesignSpecification:OutdoorAir,\n"
        "    Apt305_OA, Flow/Area, 0.002, , , , Always1;"
    )

    parts.append(
        "  ZoneHVAC:IdealLoadsAirSystem,\n"
        "    Apt305_IdealLoads, ,\n"
        "    Apt305_SupplyAir, Apt305_ExhaustAir, ,\n"
        "    50, 12, 0.010, 0.010,\n"
        "    NoLimit, , , NoLimit, , ,\n"
        "    , ,\n"
        "    ConstantSupplyHumidityRatio, ,\n"
        "    ConstantSupplyHumidityRatio,\n"
        "    Apt305_OA, , , , , , ;\n\n"
        "  ZoneHVAC:EquipmentList,\n"
        "    Apt305_EqpList,\n"
        "    SequentialLoad,\n"
        "    ZoneHVAC:IdealLoadsAirSystem, Apt305_IdealLoads, 1, 1;\n\n"
        "  ZoneHVAC:EquipmentConnections,\n"
        "    Apt305, Apt305_EqpList,\n"
        "    Apt305_SupplyAir, Apt305_ExhaustAir,\n"
        "    Apt305_ZoneAirNode;\n\n"
        "  NodeList, Apt305_SupplyAir, Apt305_SupplyAir_Node;"
    )

    parts.append(
        "  ThermostatSetpoint:DualSetpoint, Apt305_DualSP, Heat_SP, Cool_SP;\n\n"
        "  ZoneControl:Thermostat,\n"
        "    Apt305_Thermostat, Apt305, Always1,\n"
        "    ThermostatSetpoint:DualSetpoint, Apt305_DualSP;"
    )

    parts.append(
        "  OutputControl:Table:Style, HTML;\n"
        "  Output:Table:SummaryReports, AllSummary;\n"
        "  Output:Variable, Apt305, Zone Ideal Loads Heating Energy, Hourly;\n"
        "  Output:Variable, Apt305, Zone Ideal Loads Cooling Energy, Hourly;\n"
        "  Output:Diagnostics, DisplayAllWarnings;"
    )

    return "\n\n".join(parts)


In [ ]:
# ── Step 1: EnergyPlus ground-truth simulation ────────────────────────────────
IDF_PATH   = "/content/cal_apt305.idf"
EP_OUT_DIR = "/content/ep_cal_output"
REF_CSV    = "/content/ep_ref_apt305.csv"

idf_text = _make_idf()
with open(IDF_PATH, "w") as f: f.write(idf_text)

if os.path.exists(EP_OUT_DIR): shutil.rmtree(EP_OUT_DIR)
os.makedirs(EP_OUT_DIR)

print("Running EnergyPlus ...", end=" ", flush=True)
r = subprocess.run(
    [os.path.join(EP_DIR, "energyplus"), "-w", EPW_PATH,
     "-d", EP_OUT_DIR, "-p", "apt305", IDF_PATH],
    capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError(f"EnergyPlus failed:\n{r.stderr[-500:]}")
print("OK")

# ── Parse .eso → DataFrame ────────────────────────────────────────────────────
def parse_eso_hourly(eso_path,
                     tv="Zone Mean Air Temperature",
                     hv="Zone Ideal Loads Supply Air Total Heating Energy",
                     cv="Zone Ideal Loads Supply Air Total Cooling Energy"):
    var_ids = {}; rows_T, rows_H, rows_C = [], [], []
    in_dict = True
    with open(eso_path) as f:
        for line in f:
            line = line.rstrip()
            if in_dict:
                if line.strip() == "End of Data Dictionary":
                    in_dict = False; continue
                m = re.match(r'^ *(\d+),\d+,.*?,(.+?)\s*\[', line)
                if m:
                    n = m.group(2).strip()
                    if tv.lower() in n.lower():   var_ids[m.group(1)] = "T"
                    elif hv.lower() in n.lower(): var_ids[m.group(1)] = "H"
                    elif cv.lower() in n.lower(): var_ids[m.group(1)] = "C"
            else:
                parts = line.split(",", 1)
                if len(parts) == 2 and parts[0].strip() in var_ids:
                    try:
                        v = float(parts[1])
                        t = var_ids[parts[0].strip()]
                        if t == "T": rows_T.append(v)
                        elif t == "H": rows_H.append(v)
                        else: rows_C.append(v)
                    except ValueError: pass

    n = min(len(rows_T), len(rows_H), len(rows_C))
    df = pd.DataFrame({
        "T_air_C": rows_T[:n],
        "Q_H_J":   rows_H[:n],
        "Q_C_J":   rows_C[:n],
    })
    return df

eso_file = os.path.join(EP_OUT_DIR, "apt305out.eso")
ep_ref = parse_eso_hourly(eso_file)
ep_ref.to_csv(REF_CSV, index=False)

ep_h_kwh = ep_ref["Q_H_J"].sum() / 3.6e6
ep_c_kwh = ep_ref["Q_C_J"].sum() / 3.6e6
print(f"EnergyPlus reference: {len(ep_ref)} hours,  "
      f"H={ep_h_kwh:.1f} kWh,  C={ep_c_kwh:.1f} kWh")
print(f"Saved: {REF_CSV}")


In [ ]:
# ── Step 2: Candidate configurations ─────────────────────────────────────────
# Only valid pyBuildingEnergy ventilation types:
#   'occupancy' — scales with floor area × flow_rate_per_person [l/s/m²]
#   'custom'    — explicit H_ve override [W/K]
#   'temp_wind' — temperature + wind driven (ACH parameter)
#
# Note: 'eplus_infiltration_ext_area' is NOT supported by the original engine.

def set_vent_occupancy(flow):
    def fn(b):
        v = b["building_parameters"]["ventilation"]
        v["ventilation_type"]    = "occupancy"
        v["flow_rate_per_person"] = float(flow)
    return fn

def set_vent_custom(h_ve_wk):
    def fn(b):
        v = b["building_parameters"]["ventilation"]
        v["ventilation_type"] = "custom"
        v["custom_heat_transfer_coefficient_ventilation"] = float(h_ve_wk)
    return fn

def set_vent_temp_wind(ach):
    def fn(b):
        v = b["building_parameters"]["ventilation"]
        v["ventilation_type"] = "temp_wind"
        # ACH stored as flow_rate_per_person for temp_wind (engine interprets it as ACH)
        v["flow_rate_per_person"] = float(ach)
    return fn

CANDIDATES = [
    #  name                description                              modifier
    ("occ_0p3",   "Occupancy 0.3 l/s/m² (very tight)",            set_vent_occupancy(0.3)),
    ("occ_0p5",   "Occupancy 0.5 l/s/m²",                         set_vent_occupancy(0.5)),
    ("occ_0p8",   "Occupancy 0.8 l/s/m²",                         set_vent_occupancy(0.8)),
    ("occ_1p5",   "Occupancy 1.5 l/s/m²",                         set_vent_occupancy(1.5)),
    ("occ_2p0",   "Occupancy 2.0 l/s/m² (default)",               set_vent_occupancy(2.0)),
    ("occ_3p0",   "Occupancy 3.0 l/s/m²",                         set_vent_occupancy(3.0)),
    ("custom_20", "Custom H_ve = 20 W/K",                          set_vent_custom(20.0)),
    ("custom_40", "Custom H_ve = 40 W/K",                          set_vent_custom(40.0)),
    ("custom_60", "Custom H_ve = 60 W/K",                          set_vent_custom(60.0)),
    ("custom_80", "Custom H_ve = 80 W/K",                          set_vent_custom(80.0)),
    ("custom_100","Custom H_ve = 100 W/K",                         set_vent_custom(100.0)),
]

print(f"Defined {len(CANDIDATES)} candidate configurations")
for name, desc, _ in CANDIDATES:
    print(f"  {name:12s}  {desc}")


In [ ]:
# ── Step 3: Score each candidate against EnergyPlus reference ────────────────
SUMMER_IDX = set(range(0, 31*24)) | set(range((31+28)*24, (31+28+31)*24)) | set(range((365-31)*24, 365*24))  # Jan + Feb + Dec

def run_iso_candidate(modifier_fn):
    bui = build_bui()
    modifier_fn(bui)
    bui_fixed, _ = orig_sanitize(bui, fix=True)
    out = OrigISO52016.Temperature_and_Energy_needs_calculation(
        bui_fixed, weather_source=WEATHER_SOURCE)
    hourly = out[0] if len(out) == 3 else out[0]
    return hourly

def score_vs_ep(hourly_iso, ep_ref_df):
    n = min(len(hourly_iso), len(ep_ref_df))
    qhc = hourly_iso["Q_HC"].values[:n]
    ep_h_j = ep_ref_df["Q_H_J"].values[:n]
    ep_c_j = ep_ref_df["Q_C_J"].values[:n]

    # ISO energy (W each hour → Wh)
    iso_h_kwh = qhc[qhc > 0].sum() / 1000.0
    iso_c_kwh = -qhc[qhc < 0].sum() / 1000.0
    ep_h_kwh  = ep_h_j.sum() / 3.6e6
    ep_c_kwh  = ep_c_j.sum() / 3.6e6

    heat_err_pct = 100.0 * abs(iso_h_kwh - ep_h_kwh) / max(ep_h_kwh, 1.0)
    cool_err_pct = 100.0 * abs(iso_c_kwh - ep_c_kwh) / max(ep_c_kwh, 1.0)

    # Composite score (lower is better)
    score = heat_err_pct + 0.5 * cool_err_pct

    return {
        "iso_heat_kwh":   iso_h_kwh,
        "iso_cool_kwh":   iso_c_kwh,
        "ep_heat_kwh":    ep_h_kwh,
        "ep_cool_kwh":    ep_c_kwh,
        "heat_err_pct":   heat_err_pct,
        "cool_err_pct":   cool_err_pct,
        "score":          score,
    }

print("Running calibration ...\n")
results = []
for cand_name, cand_desc, cand_fn in CANDIDATES:
    print(f"  {cand_name:12s} ...", end=" ", flush=True)
    try:
        hourly = run_iso_candidate(cand_fn)
        m = score_vs_ep(hourly, ep_ref)
        m.update({"candidate": cand_name, "description": cand_desc, "status": "OK"})
        results.append(m)
        print(f"score={m['score']:5.1f}  ΔH={m['heat_err_pct']:5.1f}%  ΔC={m['cool_err_pct']:5.1f}%")
    except Exception as e:
        results.append({"candidate": cand_name, "description": cand_desc,
                        "status": f"FAILED: {e}", "score": float("inf")})
        print(f"FAILED: {e}")

ok = [r for r in results if r.get("status") == "OK"]
if not ok:
    raise RuntimeError("All candidates failed — check ventilation type names above")

results_df = pd.DataFrame(ok).sort_values("score").reset_index(drop=True)

print("\n=== Calibration ranking ===")
cols = ["candidate","description","score","heat_err_pct","cool_err_pct",
        "iso_heat_kwh","iso_cool_kwh","ep_heat_kwh","ep_cool_kwh"]
print(results_df[cols].to_string(index=False, float_format="%.1f"))

best = results_df.iloc[0]
print(f"\n✅ BEST: {best['candidate']}  —  {best['description']}")
print(f"   Heating error: {best['heat_err_pct']:.1f}%  |  Cooling error: {best['cool_err_pct']:.1f}%")
print(f"   ISO H={best['iso_heat_kwh']:.1f} kWh  vs  EP H={best['ep_heat_kwh']:.1f} kWh")
print(f"   ISO C={best['iso_cool_kwh']:.1f} kWh  vs  EP C={best['ep_cool_kwh']:.1f} kWh")
results_df.to_csv("/content/calibration_results.csv", index=False)
print("\nSaved: /content/calibration_results.csv")


In [ ]:
# ── Step 4: Summary chart — all candidates vs EnergyPlus ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

ep_h = results_df["ep_heat_kwh"].iloc[0]
ep_c = results_df["ep_cool_kwh"].iloc[0]

x = np.arange(len(results_df))
names = results_df["candidate"].tolist()
scores= results_df["score"].tolist()
colors = ["#2ecc71" if i == 0 else "#3498db" for i in range(len(results_df))]

# Score plot
axes[0].barh(x, scores, color=colors)
axes[0].set_yticks(x); axes[0].set_yticklabels(names, fontsize=8)
axes[0].set_xlabel("Composite score (lower = better)"); axes[0].set_title("Calibration score")
axes[0].axvline(0, color="k", lw=0.5)

# Heating error
h_errs = results_df["heat_err_pct"].tolist()
axes[1].barh(x, h_errs, color=colors)
axes[1].set_yticks(x); axes[1].set_yticklabels(names, fontsize=8)
axes[1].set_xlabel("Heating error %"); axes[1].set_title(f"Heating error vs EP ({ep_h:.0f} kWh)")

# Cooling error
c_errs = results_df["cool_err_pct"].tolist()
axes[2].barh(x, c_errs, color=colors)
axes[2].set_yticks(x); axes[2].set_yticklabels(names, fontsize=8)
axes[2].set_xlabel("Cooling error %"); axes[2].set_title(f"Cooling error vs EP ({ep_c:.0f} kWh)")

for ax in axes:
    ax.invert_yaxis()
plt.suptitle("ISO 52016 Calibration — Apt 305, Melbourne", fontsize=11, y=1.01)
plt.tight_layout(); plt.show()

print(f"\nBest candidate: {results_df.iloc[0]['candidate']}")
print(f"Composite score: {results_df.iloc[0]['score']:.1f}  "
      f"(baseline occ_2p0 score: {results_df[results_df.candidate=='occ_2p0']['score'].values[0] if 'occ_2p0' in results_df.candidate.values else 'N/A'})")
